# Standardized Robustness Pipeline: Poisoning + Evasion (Label Flip, HSJ, Boundary, ZOO)

This notebook runs a standardized robustness pipeline across several models.

## Pipeline steps

1. **Load dataset**
   - Read the CSV, separate features and labels, drop non-feature columns.

2. **Create one canonical split**
   - Make a single train/test split once.
   - The test split is never used for training.

3. **Baseline training + evaluation**
   - Train each model on the clean train set.
   - Evaluate clean accuracy on the held-out test set.

4. **Poisoning (train only)**
   - Apply label-flip poisoning to **training labels only** for multiple flip rates.
   - Train poisoned models and evaluate on the same fixed test set.

5. **ART detectors** (Currently not Included)
   - Wrap models/data for ART.
   - Run `art.defences.detector.poison` and `art.defences.detector.evasion` detectors where enabled.
   - Use detector outputs to filter/flag suspicious samples, then retrain and re-evaluate.

6. **Adversarial training (rounds)**
   - For each round:
     1. Train on the current train set.
     2. Generate adversarial examples from **train only** using ART against the current model.
     3. Append adversarial rows (with correct labels) to the train set.
     4. Retrain and re-evaluate.
   - Adversarial examples are regenerated each round.

7. **Robustness evaluation under attack**
   - Generate adversarial examples on a subset of the test set using ART attacks (e.g., HSJ, ZOO; Boundary skipped for NN).
   - Record clean accuracy, adversarial accuracy, and accuracy drop.

8. **Results table**
   - All metrics are collected into one dataframe (`results_df`) with columns for model, phase, round, attack, and accuracies.

In [50]:
# ===== Imports =====
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Your model runners (expected to exist in your project)
# NOTE: If running this notebook outside your project root, set PYTHONPATH accordingly.
from MachineLearning.LogReg.LogisticRegression_ML import run_best_model as run_logreg
from MachineLearning.NeuralNetworks.NeuralNet_ML import run_best_model as run_neuralnet
from MachineLearning.RandomForest.RandomForest_ML import run_best_model as run_randomforest
from MachineLearning.SVM.SVM_ML import run_best_model as run_svm
from MachineLearning.XGBoost.XGBoost_ML import run_best_model as run_xgboost

# ART: import SklearnClassifier in a way that avoids importing KerasClassifier
# (some ART installs break if keras-related utils are missing).
try:
    from art.estimators.classification.scikitlearn import SklearnClassifier
except Exception:
    from art.estimators.classification import SklearnClassifier

from art.attacks.evasion import HopSkipJump, BoundaryAttack, ZooAttack

RNG = np.random.default_rng(42)

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names",
    category=UserWarning,
)

In [ ]:
# ===== Configuration =====
DATASET_PATH = "CSVs\\dataset.csv"  # change if needed
LABEL_COL = "anomaly"

# Columns often present in your project; will be dropped from features if they exist
DROP_COLS = {"anomaly", "timestamp", "channel", "label", "segment", "train"}

# Train/test split (canonical, used for ALL models & attacks)
TEST_SIZE = 0.2
SPLIT_RANDOM_STATE = 42

# Attack budget knobs (keep realistic)
EVAL_ATTACK_SAMPLES = 200      # number of test points to attack per model/attack
TRAIN_ADV_SAMPLES = 500        # number of train points to adversarially augment per round
ROUNDS = 2                     # adversarial training rounds (Option B)

# Label flip poisoning knobs
LABEL_FLIP_RATES = [0.05, 0.10, 0.20]

# Attack configs
HSJ_KWARGS = dict(
    max_iter=15,      # 15
    max_eval=1500,    # down from 5000
    init_eval=25,     # down from 50
    init_size=200,    # down from 1000
    targeted=False,
    norm=2
)

HSJ_TRAIN_KWARGS = dict(
    max_iter=8,       # 10 -> 8
    max_eval=300,     # keep
    init_eval=25,     # keep
    init_size=200,    # 1000 -> 200
    targeted=False,
    norm=2
)

BOUNDARY_KWARGS = dict(
    targeted=False,
    max_iter=200,     # 1000 -> 200
    init_size=10
)
ZOO_KWARGS = dict(
    max_iter=5, 
    binary_search_steps=1, 
    nb_parallel=1, 
    batch_size=1
)


# ===== Detection + Retraining (Poison Robustness) =====
# DETECTORS: pick one or more. "LOSS_FILTER" is a built-in fallback that works without ART.
DETECTORS = ["LOSS_FILTER"]  # e.g., ["LOSS_FILTER", "ART_SPECTRAL"] if available in your ART install

# How aggressively to filter suspected poison points (LOSS_FILTER):
LOSS_FILTER_REMOVE_FRAC = 0.10  # remove top 10% highest-loss points from the poisoned training set

# Retraining rounds after filtering (usually 1 is enough for label-flip)
RETRAIN_ROUNDS = 1


# Evasion detectors (run on adversarial examples at test-time)
# Use names like "ART_EVASION::<ClassName>" from art.defences.detector.evasion
EVASION_DETECTORS = []  # e.g., ["ART_EVASION::BinaryInputDetector"]

# Detector kwargs (optional). Keys are class names without prefix.
POISON_DETECTOR_KWARGS = {
    # "ActivationDefence": {...},
}
EVASION_DETECTOR_KWARGS = {
    # "BinaryInputDetector": {...},
}


In [52]:
# ===== Load dataset and create canonical split =====
assert os.path.exists(DATASET_PATH), f"Dataset not found at: {DATASET_PATH}"

df = pd.read_csv(DATASET_PATH)

# Build feature columns (keep only non-label columns; drop known metadata columns if present)
feature_cols = [c for c in df.columns if c not in DROP_COLS and c != LABEL_COL]

# Keep only numeric features for attacks; if you have categorical columns, encode them before this notebook.
X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df[LABEL_COL].to_numpy(dtype=int)

# Ensure labels are 0..K-1
_, y_all = np.unique(y_all, return_inverse=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=SPLIT_RANDOM_STATE, stratify=y_all
)

print("Train:", X_train.shape, y_train.shape, "classes:", len(np.unique(y_train)))
print("Test :", X_test.shape, y_test.shape)


Train: (1698, 19) (1698,) classes: 2
Test : (425, 19) (425,)


In [53]:
# ===== Helpers: build train-only CSVs for your existing model runners =====
# Your run_best_model functions read from CSV and do their own internal split.
# To enforce "never train on test", we give them TRAIN-ONLY CSVs.
# We then evaluate the returned trained pipeline on our held-out X_test/y_test.

WORK_DIR = "StandardizedRuns"
os.makedirs(WORK_DIR, exist_ok=True)

def make_train_df_from_arrays(X_tr: np.ndarray, y_tr: np.ndarray) -> pd.DataFrame:
    out = pd.DataFrame(X_tr, columns=feature_cols)
    out[LABEL_COL] = y_tr
    return out

def save_train_csv(df_train: pd.DataFrame, name: str) -> str:
    path = os.path.join(WORK_DIR, name)
    df_train.to_csv(path, index=False)
    return path

def fit_model_with_runner(model_name: str, runner_fn, train_csv_path: str):
    # Use a more reasonable internal split than some module defaults.
    # Many of your modules default to test_size=0.80 (very large). Override to 0.2.
    pipe, _internal_X_test, _internal_y_test = runner_fn(
        path=train_csv_path,
        test_size=0.2,
        random_state=SPLIT_RANDOM_STATE
    )
    return pipe

def eval_clean(pipe, X: np.ndarray, y: np.ndarray) -> float:
    y_pred = pipe.predict(X)
    return float(accuracy_score(y, y_pred))


In [54]:
# ===== ART helpers (no detectors) =====




def wrap_art(pipe, X_ref: np.ndarray) -> SklearnClassifier:
    # clip_values: pragmatic min/max bound from training data
    clip_values = (float(np.min(X_ref)), float(np.max(X_ref)))
    return SklearnClassifier(model=pipe, clip_values=clip_values)

def predict_labels_art(art_clf: SklearnClassifier, X: np.ndarray) -> np.ndarray:
    preds = np.asarray(art_clf.predict(X))
    if preds.ndim == 1:
        return preds.astype(int)
    return np.argmax(preds, axis=1)

def sample_subset(X: np.ndarray, y: np.ndarray, n: int, rng=RNG):
    if n >= len(X):
        return X, y
    idx = rng.choice(len(X), size=n, replace=False)
    return X[idx], y[idx]

def attack_success_rate(y_true: np.ndarray, y_pred_clean: np.ndarray, y_pred_adv: np.ndarray) -> float:
    mask = (y_pred_clean == y_true)
    if mask.sum() == 0:
        return float('nan')
    return float((y_pred_adv[mask] != y_true[mask]).mean())

def eval_under_attack(attack_name: str, art_clf: SklearnClassifier, X_eval: np.ndarray, y_eval: np.ndarray):
    # Generate adversarial examples for evaluation subset and compute metrics.
    if attack_name == "HSJ":
        atk = HopSkipJump(classifier=art_clf, **HSJ_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)
    elif attack_name == "Boundary":
        # Skip Boundary for NN (sklearn MLP) because BoundaryAttack can produce NaNs
        try:
            m = getattr(art_clf, "model", None)

            # unwrap common wrapper
            if hasattr(m, "base_model"):
                m = m.base_model

            # unwrap sklearn Pipeline -> final estimator
            if hasattr(m, "steps") and len(m.steps) > 0:
                m = m.steps[-1][1]

            if m is not None and m.__class__.__name__ == "MLPClassifier":
                return np.nan, np.nan, np.nan, np.nan, {}
        except Exception:
            pass

        atk = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
        X_adv = atk.generate(x=X_eval, y=y_eval)

    elif attack_name == "ZOO":
        atk = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
        # ZOO often expects one-hot labels in some setups, but can work with integers depending on estimator.
        # We'll try integer labels first; if it errors, we fall back to simple one-hot.
        try:
            X_adv = atk.generate(x=X_eval, y=y_eval)
        except Exception:
            k = int(len(np.unique(y_train)))
            y_oh = np.zeros((len(y_eval), k), dtype=np.float32)
            y_oh[np.arange(len(y_eval)), y_eval.astype(int)] = 1.0
            X_adv = atk.generate(x=X_eval, y=y_oh)
    else:
        raise ValueError(f"Unknown attack: {attack_name}")

    X_adv = np.asarray(X_adv, dtype=np.float32)

    y_pred_clean = predict_labels_art(art_clf, X_eval)
    y_pred_adv = predict_labels_art(art_clf, X_adv)


    ev_det_metrics = run_evasion_detectors_on_adv(X_adv, y_pred_adv)
    clean_acc = float(accuracy_score(y_eval, y_pred_clean))
    adv_acc = float(accuracy_score(y_eval, y_pred_adv))
    drop = clean_acc - adv_acc
    asr = attack_success_rate(y_eval, y_pred_clean, y_pred_adv)
    return clean_acc, adv_acc, drop, asr, ev_det_metrics


def _try_import_art_detectors():
    """Best-effort import for ART detector modules. Returns (poison_mod, evasion_mod) or (None, None)."""
    try:
        import art  # noqa: F401
        from art.defences.detector import poison as poison_mod
        from art.defences.detector import evasion as evasion_mod
        return poison_mod, evasion_mod
    except Exception:
        return None, None

def list_art_detector_classes():
    """List available class names in art.defences.detector.poison and .evasion (if installed)."""
    poison_mod, evasion_mod = _try_import_art_detectors()
    out = {"poison": [], "evasion": []}
    import inspect
    if poison_mod is not None:
        for name, obj in vars(poison_mod).items():
            if inspect.isclass(obj) and obj.__module__.startswith("art."):
                out["poison"].append(name)
    if evasion_mod is not None:
        for name, obj in vars(evasion_mod).items():
            if inspect.isclass(obj) and obj.__module__.startswith("art."):
                out["evasion"].append(name)
    out["poison"].sort()
    out["evasion"].sort()
    return out

def _instantiate_art_class(mod, class_name: str, kwargs: dict, *args):
    """Instantiate class from an ART module, filtering kwargs to match the constructor signature."""
    import inspect
    cls = getattr(mod, class_name, None)
    if cls is None:
        raise ImportError(f"ART class not found: {class_name}")
    sig = None
    try:
        sig = inspect.signature(cls.__init__)
    except Exception:
        sig = None
    if sig is not None:
        valid = {k: v for k, v in (kwargs or {}).items() if k in sig.parameters}
    else:
        valid = kwargs or {}
    return cls(*args, **valid)

def run_evasion_detectors_on_adv(X_adv: np.ndarray, y_pred_adv: np.ndarray):
    """Run configured EVASION_DETECTORS and return dict of metrics."""
    metrics = {}
    if not EVASION_DETECTORS:
        return metrics

    _, evasion_mod = _try_import_art_detectors()
    if evasion_mod is None:
        for det in EVASION_DETECTORS:
            metrics[f"evasion_det::{det}"] = np.nan
        return metrics

    import numpy as np
    for det_name in EVASION_DETECTORS:
        if not det_name.startswith("ART_EVASION::"):
            continue
        class_name = det_name.split("::", 1)[1]
        kwargs = EVASION_DETECTOR_KWARGS.get(class_name, {}) if "EVASION_DETECTOR_KWARGS" in globals() else {}
        try:
            det = _instantiate_art_class(evasion_mod, class_name, kwargs)
            # Try common APIs
            if hasattr(det, "detect"):
                out = det.detect(X_adv)  # some detectors take only x
            elif hasattr(det, "predict"):
                out = det.predict(X_adv)
            else:
                raise AttributeError("Detector has no detect/predict method")
            out = np.asarray(out).ravel()
            # Interpret: if boolean -> True means detected; if scores -> threshold at 0.5 (best-effort)
            if out.dtype == bool:
                detected = out
            else:
                detected = out > 0.5
            metrics[f"evasion_det::{class_name}::detected_rate"] = float(np.mean(detected))
        except Exception:
            metrics[f"evasion_det::{class_name}::detected_rate"] = np.nan
    return metrics


In [55]:
# ===== Poison detection + retraining helpers =====
# Supports built-in LOSS_FILTER and best-effort ART poison detectors from art.defences.detector.poison
# Goal: Train -> Poison -> Detect (on poisoned train) -> Retrain -> Test

def per_sample_log_loss_from_proba(y_true: np.ndarray, proba: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Cross-entropy / log-loss per sample given predicted probabilities."""
    y_true = np.asarray(y_true, dtype=int)
    proba = np.asarray(proba, dtype=np.float64)
    proba = np.clip(proba, eps, 1.0 - eps)
    if proba.ndim == 1:
        # binary edge case: proba = P(class=1)
        proba = np.stack([1.0 - proba, proba], axis=1)
    return -np.log(proba[np.arange(len(y_true)), y_true])

def loss_filter_detector(pipe, X_train: np.ndarray, y_train: np.ndarray, remove_frac: float) -> np.ndarray:
    """Return a boolean mask of points to KEEP (True = keep)."""
    # works for sklearn-like pipelines with predict_proba
    if not hasattr(pipe, "predict_proba"):
        # fallback: use ART prediction (already handles logits/onehot)
        proba = pipe.predict(X_train)
    else:
        proba = pipe.predict_proba(X_train)

    losses = per_sample_log_loss_from_proba(y_train, proba)
    n = len(losses)
    k = int(max(1, round(remove_frac * n)))
    worst_idx = np.argsort(losses)[-k:]  # highest loss = most suspicious for label-noise/flip
    keep = np.ones(n, dtype=bool)
    keep[worst_idx] = False
    return keep

def try_art_poison_detector(detector_name: str, art_clf, X_train: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    """Optional: use ART's poisoning detectors if available.
    Returns keep-mask (True = keep). If detector isn't available, raises ImportError/AttributeError.
    """
    name = detector_name.upper()

    # NOTE: ART detector class names vary by version; these imports are best-effort.
    # If your version differs, search your ART docs for 'poison' defenses and adapt here.
    if name in ["ART_SPECTRAL", "SPECTRAL", "SPECTRAL_SIGNATURE"]:
        try:
            from art.defences.detector.poison import SpectralSignatureDefense
        except Exception as e:
            raise ImportError("SpectralSignatureDefense import failed. Check your ART version.") from e

        # SpectralSignatureDefense API differs across versions. This is a common pattern:
        defence = SpectralSignatureDefense(classifier=art_clf, x_train=X_train, y_train=y_train)
        report = defence.detect_poison()
        # Common conventions: report might include 'is_clean' or 'poisonous_indices'
        if isinstance(report, dict) and "is_clean" in report:
            return np.asarray(report["is_clean"], dtype=bool)
        if isinstance(report, dict) and "poisonous_indices" in report:
            keep = np.ones(len(X_train), dtype=bool)
            keep[np.asarray(report["poisonous_indices"], dtype=int)] = False
            return keep
        raise RuntimeError("Unexpected SpectralSignatureDefense output; inspect `report` and adapt.")

    elif name in ["ART_ACTIVATION", "ACTIVATION_DEFENCE", "ACTIVATION"]:
        try:
            from art.defences.detector.poison import ActivationDefence
        except Exception as e:
            raise ImportError("ActivationDefence import failed. Check your ART version.") from e

        defence = ActivationDefence(classifier=art_clf, x_train=X_train, y_train=y_train)
        report = defence.detect_poison()
        if isinstance(report, dict) and "is_clean" in report:
            return np.asarray(report["is_clean"], dtype=bool)
        if isinstance(report, dict) and "poisonous_indices" in report:
            keep = np.ones(len(X_train), dtype=bool)
            keep[np.asarray(report["poisonous_indices"], dtype=int)] = False
            return keep
        raise RuntimeError("Unexpected ActivationDefence output; inspect `report` and adapt.")
    else:
        raise ValueError(f"Unknown detector_name: {detector_name}")

def detect_poison(detector_name: str, pipe, art_clf, X_train: np.ndarray, y_train: np.ndarray, remove_frac: float) -> np.ndarray:
    """Unified detection interface returning keep-mask."""
    if detector_name.upper() == "LOSS_FILTER":
        return loss_filter_detector(pipe, X_train, y_train, remove_frac)
    else:
        return try_art_poison_detector(detector_name, art_clf, X_train, y_train)

def score_detection(keep_mask: np.ndarray, poison_indices: list[int]) -> dict:
    """Compute simple detection metrics if we know the poison indices."""
    n = len(keep_mask)
    poison = np.zeros(n, dtype=bool)
    poison[np.asarray(poison_indices, dtype=int)] = True

    predicted_poison = ~keep_mask

    tp = int(np.sum(predicted_poison & poison))
    fp = int(np.sum(predicted_poison & ~poison))
    fn = int(np.sum((~predicted_poison) & poison))
    tn = int(np.sum((~predicted_poison) & (~poison)))

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else np.nan

    return {
        "det_tp": tp, "det_fp": fp, "det_fn": fn, "det_tn": tn,
        "det_precision": float(precision) if precision==precision else np.nan,
        "det_recall": float(recall) if recall==recall else np.nan,
        "det_f1": float(f1) if f1==f1 else np.nan,
        "det_flagged_frac": float(np.mean(predicted_poison)),
    }


def run_art_poison_detector(detector_name: str, art_clf: SklearnClassifier, X_train: np.ndarray, y_train: np.ndarray) -> np.ndarray:
    """Return a boolean keep mask using an ART poison detector (best-effort, API varies by ART version)."""
    import numpy as np

    poison_mod, _ = _try_import_art_detectors()
    if poison_mod is None:
        raise ImportError("ART is not installed or art.defences.detector.poison is unavailable in this environment.")

    class_name = detector_name.split("::", 1)[1]
    kwargs = POISON_DETECTOR_KWARGS.get(class_name, {}) if "POISON_DETECTOR_KWARGS" in globals() else {}

    det = _instantiate_art_class(poison_mod, class_name, kwargs, art_clf)

    # Try common methods / return patterns
    suspected = None

    # 1) detect_poison(x, y) -> indices / mask / dict
    if hasattr(det, "detect_poison"):
        out = det.detect_poison(X_train, y_train)
        suspected = out

    # 2) detect(x, y) -> indices / mask / scores
    elif hasattr(det, "detect"):
        try:
            suspected = det.detect(X_train, y_train)
        except TypeError:
            suspected = det.detect(X_train)

    # 3) mitigate(x, y) -> cleaned_x, cleaned_y, report
    elif hasattr(det, "mitigate"):
        out = det.mitigate(X_train, y_train)
        suspected = out

    # Normalize output into keep_mask
    n = len(X_train)
    keep_mask = np.ones(n, dtype=bool)

    def _mask_from_indices(idxs):
        m = np.ones(n, dtype=bool)
        m[np.asarray(idxs, dtype=int)] = False
        return m

    if suspected is None:
        raise RuntimeError(f"{class_name}: could not run detector (no supported method found).")

    # If output is tuple, try to find indices-like thing
    if isinstance(suspected, tuple) or isinstance(suspected, list):
        # Find any 1D int-like array
        for item in suspected:
            arr = np.asarray(item)
            if arr.ndim == 1 and arr.size <= n:
                # could be indices or boolean mask
                if arr.dtype == bool and arr.size == n:
                    keep_mask = arr
                    return keep_mask
                # indices heuristic: integer + within range
                if np.issubdtype(arr.dtype, np.integer):
                    keep_mask = _mask_from_indices(arr)
                    return keep_mask
        # If we got cleaned arrays, infer kept indices by matching (fallback: keep all)
        return keep_mask

    # If dict-like
    if isinstance(suspected, dict):
        # common keys
        for k in ["suspected_poison", "poison_indices", "indices", "detected_poison"]:
            if k in suspected:
                arr = np.asarray(suspected[k])
                if arr.dtype == bool and arr.size == n:
                    return arr
                if np.issubdtype(arr.dtype, np.integer):
                    return _mask_from_indices(arr)
        # scores key
        for k in ["scores", "score", "anomaly_score"]:
            if k in suspected:
                scores = np.asarray(suspected[k]).ravel()
                if scores.size == n:
                    # remove top fraction if user provided remove_frac, else 10%
                    frac = float(kwargs.get("remove_frac", LOSS_FILTER_REMOVE_FRAC if "LOSS_FILTER_REMOVE_FRAC" in globals() else 0.10))
                    k_remove = int(np.ceil(frac * n))
                    bad = np.argsort(scores)[-k_remove:]
                    return _mask_from_indices(bad)
        return keep_mask

    # ndarray
    arr = np.asarray(suspected)
    if arr.dtype == bool and arr.size == n:
        return arr
    if np.issubdtype(arr.dtype, np.integer):
        return _mask_from_indices(arr)
    if arr.size == n:
        # treat as scores
        frac = float(kwargs.get("remove_frac", LOSS_FILTER_REMOVE_FRAC if "LOSS_FILTER_REMOVE_FRAC" in globals() else 0.10))
        k_remove = int(np.ceil(frac * n))
        bad = np.argsort(arr.ravel())[-k_remove:]
        return _mask_from_indices(bad)

    return keep_mask

def detect_poison(detector_name: str, pipe, art_clf: SklearnClassifier, X_train: np.ndarray, y_train: np.ndarray, poison_indices=None) -> dict:
    """Run poison detector and return dict with keep_mask + detection metrics."""
    import numpy as np

    if detector_name == "LOSS_FILTER":
        keep_mask = loss_filter_detector(pipe, X_train, y_train, remove_frac=LOSS_FILTER_REMOVE_FRAC)
    elif detector_name.startswith("ART_POISON::"):
        keep_mask = run_art_poison_detector(detector_name, art_clf, X_train, y_train)
    else:
        raise ValueError(f"Unknown detector: {detector_name}")

    out = {"keep_mask": keep_mask}
    if poison_indices is not None and len(poison_indices) > 0:
        out.update(score_detection(keep_mask, poison_indices))
    return out


In [56]:
# ===== Poisoning: Label Flip (train-only) =====

def label_flip(y: np.ndarray, flip_rate: float, num_classes: int, rng=RNG) -> tuple[np.ndarray, dict]:
    """Flip labels for a subset of training points. Returns (y_poisoned, meta).

    meta["indices"] are the positions in y that were flipped.
    """
    y = np.asarray(y, dtype=int).copy()
    n = len(y)
    k = int(num_classes)
    m = int(round(flip_rate * n))
    idx = rng.choice(n, size=m, replace=False)

    if k == 2:
        y[idx] = 1 - y[idx]
    else:
        for i in idx:
            choices = [c for c in range(k) if c != y[i]]
            y[i] = rng.choice(choices)

    meta = {"flip_rate": float(flip_rate), "num_flipped": int(m), "indices": idx.tolist()}
    return y, meta


In [57]:
# ===== Standardized experiment runners =====

MODEL_RUNNERS = {
    "LogReg": run_logreg,
    "NeuralNet": run_neuralnet,
    "RandomForest": run_randomforest,
    "SVM": run_svm,
    # "XGBoost": run_xgboost,
}

EVASION_ATTACKS = ["HSJ", "Boundary", "ZOO"]

def _flatten_metrics(d: dict, prefix: str = ""):
    if not d:
        return {}
    return {f"{prefix}{k}": v for k, v in d.items()}


def run_baseline_and_evasion(model_name: str, runner_fn):
    rows = []

    # Train on clean TRAIN ONLY
    train_df = make_train_df_from_arrays(X_train, y_train)
    clean_train_path = save_train_csv(train_df, f"{model_name}_train_clean.csv")
    pipe = fit_model_with_runner(model_name, runner_fn, clean_train_path)

    # Clean eval (full held-out test)
    clean_test_acc = eval_clean(pipe, X_test, y_test)

    # Evasion eval on subset
    art_clf = wrap_art(pipe, X_train)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_clf, X_eval, y_eval)
        rows.append({
            "model": model_name,
            "phase": "baseline",
            "attack_type": "evasion",
            "attack": atk,
            "round": 0,
            "clean_test_acc": clean_test_acc,
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": 0.0,
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
        })

    return pipe, pd.DataFrame(rows)

def run_label_flip_poisoning(model_name: str, runner_fn, flip_rate: float):
    rows = []

    num_classes = int(np.unique(y_train).size)
    y_poison, meta = label_flip(y_train, flip_rate, num_classes=num_classes)

    train_df_poison = make_train_df_from_arrays(X_train, y_poison)
    poison_path = save_train_csv(train_df_poison, f"{model_name}_train_labelflip_{int(flip_rate*100)}.csv")

    pipe = fit_model_with_runner(model_name, runner_fn, poison_path)

    clean_test_acc = eval_clean(pipe, X_test, y_test)

    # Optional: evaluate evasion robustness after poisoning (often interesting)
    art_clf = wrap_art(pipe, X_train)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_clf, X_eval, y_eval)
        rows.append({
            "model": model_name,
            "phase": "poisoned",
            "attack_type": "poison+evasion",
            "attack": f"LabelFlip({flip_rate}) + {atk}",
            "round": 0,
            "clean_test_acc": clean_test_acc,
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": float(flip_rate),
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
            "poison_meta": json.dumps({k: v for k, v in meta.items() if k != "indices"}),
        })

    return pd.DataFrame(rows)

def run_adversarial_training(model_name: str, runner_fn, train_attack: str = "HSJ"):
    rows = []

    # Start with clean train
    X_tr = X_train.copy()
    y_tr = y_train.copy()
    augmented = 0

    for r in range(ROUNDS + 1):
        # Train current model on current train set (clean + accumulated adv)
        train_df = make_train_df_from_arrays(X_tr, y_tr)
        train_path = save_train_csv(train_df, f"{model_name}_train_advtrain_round{r}.csv")
        pipe = fit_model_with_runner(model_name, runner_fn, train_path)

        # Evaluate on full clean test
        clean_test_acc = eval_clean(pipe, X_test, y_test)

        # Evaluate evasion robustness on subset (HSJ/Boundary/ZOO)
        art_clf = wrap_art(pipe, X_tr)
        X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

        for atk in EVASION_ATTACKS:
            cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_clf, X_eval, y_eval)
            rows.append({
                "model": model_name,
                "phase": "advtrain",
                "attack_type": "evasion",
                "attack": atk,
                "round": r,
                "clean_test_acc": clean_test_acc,
                "clean_acc_evalsubset": cacc,
                "adv_acc_evalsubset": aacc,
                "acc_drop_evalsubset": drop,
                "attack_success_rate": asr,
                "train_poison_rate": 0.0,
                "train_adv_augmented": augmented,
                "eval_attack_samples": len(X_eval),
            })

        if r == ROUNDS:
            break

        # Generate fresh adversarials from TRAIN subset only
        X_sub, y_sub = sample_subset(X_tr, y_tr, TRAIN_ADV_SAMPLES)

        if train_attack == "HSJ":
            atk_train = HopSkipJump(classifier=art_clf, **HSJ_TRAIN_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "Boundary":
            atk_train = BoundaryAttack(estimator=art_clf, **BOUNDARY_KWARGS)
            X_adv = atk_train.generate(x=X_sub, y=y_sub)
        elif train_attack == "ZOO":
            atk_train = ZooAttack(classifier=art_clf, **ZOO_KWARGS)
            try:
                X_adv = atk_train.generate(x=X_sub, y=y_sub)
            except Exception:
                k = int(len(np.unique(y_train)))
                y_oh = np.zeros((len(y_sub), k), dtype=np.float32)
                y_oh[np.arange(len(y_sub)), y_sub.astype(int)] = 1.0
                X_adv = atk_train.generate(x=X_sub, y=y_oh)
        else:
            raise ValueError(train_attack)

        X_adv = np.asarray(X_adv, dtype=np.float32)

        # Append with correct labels
        X_tr = np.vstack([X_tr, X_adv]).astype(np.float32)
        y_tr = np.concatenate([y_tr, y_sub]).astype(int)
        augmented += len(X_adv)

        print(f"[{model_name}] adv-train round {r} -> {r+1}: +{len(X_adv)} using {train_attack}, train size={len(X_tr)}")

    return pd.DataFrame(rows)

def run_poison_detect_retrain(model_name: str, runner_fn, flip_rate: float, detector_name: str):
    """Train clean -> poison labels -> train poisoned -> detect -> retrain on filtered -> test (+ optional evasion eval)."""
    rows = []

    num_classes = int(len(np.unique(y_train)))

    # 1) Clean training (train-only CSV)
    train_df = make_train_df_from_arrays(X_train, y_train)
    clean_train_path = save_train_csv(train_df, f"{model_name}_train_clean_for_detect.csv")
    pipe_clean = fit_model_with_runner(model_name, runner_fn, clean_train_path)
    clean_test_acc = eval_clean(pipe_clean, X_test, y_test)

    # 2) Poison labels and train poisoned model
    y_poison, meta = label_flip(y_train, flip_rate=flip_rate, num_classes=num_classes)
    poison_df = make_train_df_from_arrays(X_train, y_poison)
    poison_train_path = save_train_csv(poison_df, f"{model_name}_train_poison_flip{flip_rate:.3f}.csv")
    pipe_poison = fit_model_with_runner(model_name, runner_fn, poison_train_path)
    poisoned_test_acc = eval_clean(pipe_poison, X_test, y_test)

    # 3) Detect poison points (on poisoned training set)
    art_poison = wrap_art(pipe_poison, X_train)
    keep_mask = detect_poison(detector_name, pipe_poison, art_poison, X_train, y_poison, LOSS_FILTER_REMOVE_FRAC)
    det_metrics = score_detection(keep_mask, meta["indices"])

    # 4) Retrain on filtered dataset (optionally multiple rounds)
    X_filt = X_train[keep_mask]
    y_filt = y_poison[keep_mask]

    pipe_rt = pipe_poison
    rt_test_acc = poisoned_test_acc
    for rr in range(RETRAIN_ROUNDS):
        filt_df = make_train_df_from_arrays(X_filt, y_filt)
        filt_train_path = save_train_csv(
            filt_df,
            f"{model_name}_train_filtered_{detector_name}_r{rr+1}_flip{flip_rate:.3f}.csv"
        )
        pipe_rt = fit_model_with_runner(model_name, runner_fn, filt_train_path)
        rt_test_acc = eval_clean(pipe_rt, X_test, y_test)

    # 5) Optional: evasion evaluation after retraining
    art_rt = wrap_art(pipe_rt, X_filt)
    X_eval, y_eval = sample_subset(X_test, y_test, EVAL_ATTACK_SAMPLES)

    # Summary rows (clean, poisoned, retrained)
    base_common = {k: det_metrics.get(k, np.nan) for k in det_metrics}
    rows.append({
        "model": model_name,
        "phase": "clean",
        "attack_type": "none",
        "attack": "none",
        "round": 0,
        "clean_test_acc": float(clean_test_acc),
        "clean_acc_evalsubset": np.nan,
        "adv_acc_evalsubset": np.nan,
        "acc_drop_evalsubset": np.nan,
        "attack_success_rate": np.nan,
        "train_poison_rate": 0.0,
        "detector": None,
        "train_adv_augmented": 0,
        "eval_attack_samples": len(X_eval),
        "poison_meta": None,
        **base_common,
    })
    rows.append({
        "model": model_name,
        "phase": "poisoned",
        "attack_type": "poison",
        "attack": f"LabelFlip({flip_rate})",
        "round": 0,
        "clean_test_acc": float(poisoned_test_acc),
        "clean_acc_evalsubset": np.nan,
        "adv_acc_evalsubset": np.nan,
        "acc_drop_evalsubset": np.nan,
        "attack_success_rate": np.nan,
        "train_poison_rate": float(flip_rate),
        "detector": None,
        "train_adv_augmented": 0,
        "eval_attack_samples": len(X_eval),
        "poison_meta": json.dumps({k: v for k, v in meta.items() if k != "indices"}),
        **base_common,
    })
    rows.append({
        "model": model_name,
        "phase": "detected_retrained",
        "attack_type": "poison",
        "attack": f"LabelFlip({flip_rate}) + Detect({detector_name}) + Retrain({RETRAIN_ROUNDS})",
        "round": 0,
        "clean_test_acc": float(rt_test_acc),
        "clean_acc_evalsubset": np.nan,
        "adv_acc_evalsubset": np.nan,
        "acc_drop_evalsubset": np.nan,
        "attack_success_rate": np.nan,
        "train_poison_rate": float(flip_rate),
        "detector": detector_name,
        "train_adv_augmented": 0,
        "eval_attack_samples": len(X_eval),
        "poison_meta": json.dumps({k: v for k, v in meta.items() if k != "indices"}),
        **base_common,
    })

    for atk in EVASION_ATTACKS:
        cacc, aacc, drop, asr, evm = eval_under_attack(atk, art_rt, X_eval, y_eval)
        rows.append({
            "model": model_name,
            "phase": "detected_retrained",
            "attack_type": "poison+evasion",
            "attack": f"LabelFlip({flip_rate}) + Detect({detector_name}) + Retrain({RETRAIN_ROUNDS}) + {atk}",
            "round": 0,
            "clean_test_acc": float(rt_test_acc),
            "clean_acc_evalsubset": cacc,
            "adv_acc_evalsubset": aacc,
            "acc_drop_evalsubset": drop,
            "attack_success_rate": asr,
            "train_poison_rate": float(flip_rate),
            "detector": detector_name,
            "train_adv_augmented": 0,
            "eval_attack_samples": len(X_eval),
            "poison_meta": json.dumps({k: v for k, v in meta.items() if k != "indices"}),
            **base_common,
        })

    return pd.DataFrame(rows)


In [58]:
# ===== Run the full standardized suite =====

all_rows = []

for model_name, runner_fn in MODEL_RUNNERS.items():
    print("\n" + "="*80)
    print("MODEL:", model_name)
    print("="*80)

    # Baseline + evasion attacks (clean training)
    _pipe, df_base = run_baseline_and_evasion(model_name, runner_fn)
    all_rows.append(df_base)

    # Poisoning: label flip rates (train poisoned models + evaluate)
    for rate in LABEL_FLIP_RATES:
        df_poison = run_label_flip_poisoning(model_name, runner_fn, rate)
        all_rows.append(df_poison)

        # Detection -> retrain -> evaluate
        for det in DETECTORS:
            try:
                df_det = run_poison_detect_retrain(model_name, runner_fn, flip_rate=rate, detector_name=det)
                all_rows.append(df_det)
            except Exception as e:
                print(f"[{model_name}] Detect/Retrain skipped for detector={det}, rate={rate}: {type(e).__name__}: {e}")

    # Option B: adversarial training (default HSJ) - comment out if too slow
    df_advtrain = run_adversarial_training(model_name, runner_fn, train_attack="HSJ")
    all_rows.append(df_advtrain)

results_df = pd.concat(all_rows, ignore_index=True)
results_df



MODEL: LogReg
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
New best model found! (F1 0.916 > 0.896)
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.981)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       271
           1       0.88      0.81      0.84        69

    accuracy                           0.94       340
   macro avg       0.91      0.89      0.90       340
weighted avg       0.94      0.94      0.94       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     263       8
True 1      13      56

AUC: 0.981

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 200/200 [00:00<00:00, 279.86it/s]


Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.899)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       262
           1       0.77      0.78      0.78        78

    accuracy                           0.90       340
   macro avg       0.85      0.86      0.86       340
weighted avg       0.90      0.90      0.90       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     244      18
True 1      17      61

AUC: 0.899

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 200/200 [00:00<00:00, 283.43it/s]


Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run results to Results/LogRegResults\results_logreg.csv
Saved ROC data to Results/LogRegResults\roc_logreg_clean.csv (AUC = 0.981)
Saved summary (AUC + Confusion Matrix) to Results/LogRegResults\logreg_summary.csv

=== Test Set Classification Report ===
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       271
           1       0.88      0.81      0.84        69

    accuracy                           0.94       340
   macro avg       0.91      0.89      0.90       340
weighted avg       0.94      0.94      0.94       340


Confusion Matrix:
         Pred 0  Pred 1
True 0     263       8
True 1      13      56

AUC: 0.981

=== All results and summaries saved successfully ===
Using parameters: {'clf__C': 10, 'clf__class_weight': 'balanced', 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Saved last run resu

...

[SVM] adv-train round 0 -> 1: +500 using HSJ, train size=2198

Saved last run results to Results/SVMResults\results_svm.csv
Saved ROC data to Results/SVMResults\roc_svm_clean.csv (AUC = 0.929)
Saved summary (AUC + Confusion Matrix) to Results/SVMResults\svm_summary.csv

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.93      0.98      0.96      1575
           1       0.91      0.71      0.80       404

    accuracy                           0.93      1979
   macro avg       0.92      0.85      0.88      1979
weighted avg       0.93      0.93      0.92      1979


Confusion Matrix:
         Pred 0  Pred 1
True 0    1547      28
True 1     116     288

AUC: 0.929

=== All results and summaries saved successfully ===


HopSkipJump: 100%|██████████| 500/500 [00:15<00:00, 31.86it/s]


[SVM] adv-train round 1 -> 2: +500 using HSJ, train size=2698

Saved last run results to Results/SVMResults\results_svm.csv
Saved ROC data to Results/SVMResults\roc_svm_clean.csv (AUC = 0.949)
Saved summary (AUC + Confusion Matrix) to Results/SVMResults\svm_summary.csv

=== SVM Test Set Report ===
              precision    recall  f1-score   support

           0       0.93      0.99      0.96      1940
           1       0.96      0.72      0.82       489

    accuracy                           0.94      2429
   macro avg       0.95      0.86      0.89      2429
weighted avg       0.94      0.94      0.93      2429


Confusion Matrix:
         Pred 0  Pred 1
True 0    1926      14
True 1     136     353

AUC: 0.949

=== All results and summaries saved successfully ===


ZOO: 100%|██████████| 200/200 [00:00<00:00, 230.47it/s]


,model,phase,attack_type,attack,round,clean_test_acc,clean_acc_evalsubset,adv_acc_evalsubset,acc_drop_evalsubset,attack_success_rate,train_poison_rate,train_adv_augmented,eval_attack_samples,poison_meta
0,LogReg,baseline,evasion,HSJ,0,0.931765,0.930,0.070,0.860,1.000000,0.00,0,200,NaN
1,LogReg,baseline,evasion,Boundary,0,0.931765,0.930,0.380,0.550,0.650538,0.00,0,200,NaN
2,LogReg,baseline,evasion,ZOO,0,0.931765,0.930,0.315,0.615,0.661290,0.00,0,200,NaN
3,LogReg,poisoned,poison+evasion,LabelFlip(0.05) + HSJ,0,0.912941,0.880,0.120,0.760,1.000000,0.05,0,200,"{""flip_rate"": 0.05, ""num_flipped"": 85}"
4,LogReg,poisoned,poison+evasion,LabelFlip(0.05) + Boundary,0,0.912941,0.880,0.130,0.750,0.988636,0.05,0,200,"{""flip_rate"": 0.05, ""num_flipped"": 85}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,SVM,advtrain,evasion,Boundary,1,0.920000,0.930,0.075,0.855,0.994624,0.00,500,200,NaN
80,SVM,advtrain,evasion,ZOO,1,0.920000,0.930,0.330,0.600,0.645161,0.00,500,200,NaN
81,SVM,advtrain,evasion,HSJ,2,0.912941,0.895,0.195,0.700,0.888268,0.00,1000,200,NaN
82,SVM,advtrain,evasion,Boundary,2,0.912941,0.895,0.195,0.700,0.888268,0.00,1000,200,NaN


In [59]:
# ===== Save results and quick pivots =====

out_csv = os.path.join(WORK_DIR, "standardized_results.csv")
results_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# Quick pivot: baseline evasion drops
pivot = results_df[results_df["phase"].isin(["baseline", "advtrain"])].pivot_table(
    index=["model", "phase", "round"],
    columns=["attack"],
    values=["clean_test_acc", "adv_acc_evalsubset", "acc_drop_evalsubset"],
    aggfunc="mean"
)
pivot


Saved: StandardizedRuns\standardized_results.csv


acc_drop_evalsubset                \
attack                                 Boundary    HSJ    ZOO   
model        phase    round                                     
LogReg       advtrain 0                   0.545  0.830  0.620   
                      1                   0.900  0.900  0.620   
                      2                   0.740  0.825  0.655   
             baseline 0                   0.550  0.860  0.615   
NeuralNet    advtrain 0                     NaN  0.920  0.590   
                      1                     NaN  0.910  0.505   
                      2                     NaN  0.760  0.465   
             baseline 0                     NaN  0.920  0.595   
RandomForest advtrain 0                   0.160  0.225  0.010   
                      1                   0.130  0.130  0.075   
                      2                   0.025  0.025  0.165   
             baseline 0                   0.160  0.235  0.025   
SVM          advtrain 0                   0.425  0.800  0.540   
                      1                   0.855  0.860  0.600   
                      2                   0.700  0.700  0.570   
             baseline 0                   0.410  0.810  0.540   

                            adv_acc_evalsubset               clean_test_acc  \
attack                                Boundary    HSJ    ZOO       Boundary   
model        phase    round                                                   
LogReg       advtrain 0                  0.370  0.085  0.295       0.931765   
                      1                  0.050  0.050  0.330       0.934118   
                      2                  0.185  0.100  0.270       0.936471   
             baseline 0                  0.380  0.070  0.315       0.931765   
NeuralNet    advtrain 0                    NaN  0.040  0.370       0.957647   
                      1                    NaN  0.045  0.450       0.955294   
                      2                    NaN  0.195  0.490       0.960000   
             baseline 0                    NaN  0.040  0.365       0.957647   
RandomForest advtrain 0                  0.785  0.720  0.935       0.950588   
                      1                  0.805  0.805  0.860       0.945882   
                      2                  0.790  0.790  0.650       0.816471   
             baseline 0                  0.785  0.710  0.920       0.950588   
SVM          advtrain 0                  0.475  0.100  0.360       0.924706   
                      1                  0.075  0.070  0.330       0.920000   
                      2                  0.195  0.195  0.325       0.912941   
             baseline 0                  0.495  0.095  0.365       0.924706   

                                                 
attack                            HSJ       ZOO  
model        phase    round                      
LogReg       advtrain 0      0.931765  0.931765  
                      1      0.934118  0.934118  
                      2      0.936471  0.936471  
             baseline 0      0.931765  0.931765  
NeuralNet    advtrain 0      0.957647  0.957647  
                      1      0.955294  0.955294  
                      2      0.960000  0.960000  
             baseline 0      0.957647  0.957647  
RandomForest advtrain 0      0.950588  0.950588  
                      1      0.945882  0.945882  
                      2      0.816471  0.816471  
             baseline 0      0.950588  0.950588  
SVM          advtrain 0      0.924706  0.924706  
                      1      0.920000  0.920000  
                      2      0.912941  0.912941  
             baseline 0      0.924706  0.924706